In [ ]:
import numpy as np
import pandas as pd

# 1 How to Think About Group Operations

In [ ]:
df = pd.DataFrame({
    "key1": ["a", "a", None, "b", "b", "a", None],
    "key2" : pd.Series([1, 2, 1, 2, 1, None, 1], dtype="Int64"),
    "data1" : np.random.standard_normal(7),
    "data2" : np.random.standard_normal(7)
})

df

,key1,key2,data1,data2
0,a,1,1.637544,-0.305685
1,a,2,-0.953149,-0.332670
2,None,1,0.576845,-1.431820
3,b,2,0.519965,-0.449019
4,b,1,0.306656,-0.831162
5,a,<NA>,1.001695,0.160066
6,None,1,1.359904,-0.599777


Suppose you wanted to compute the mean of the `data1` column using the labels from `key1`. There are a number of ways to do this. One is to access `data1` and call `groupby` with the column (a Series) at `key1`:

In [ ]:
grouped = df["data1"].groupby(df["key1"])

grouped

This `grouped` variable is now a special "GroupBy" object. It has not actually computed anything yet except for some intermediate data about the group key df["key1"]. The idea is that this object has all of the information needed to then apply some operation to each of the groups. For example, to compute group means we can call the GroupBy’s `mean` method:

In [ ]:
grouped.mean()

,data1
key1,
a,0.56203
b,0.41331


If instead we had passed multiple arrays as a list, we'd get something different:

In [ ]:
means = df["data1"].groupby([df["key1"], df["key2"]]).mean()

means

key1  key2
a     1       1.637544
      2      -0.953149
b     1       0.306656
      2       0.519965
Name: data1, dtype: float64

In [ ]:
means.unstack()

key2,1,2
key1,,
a,1.637544,-0.953149
b,0.306656,0.519965


In this example, the group keys are all Series, though they could be any arrays of the right length:

In [ ]:
states = np.array(["OH", "CA", "CA", "OH", "OH", "CA", "OH"])

years = [2005, 2005, 2006, 2005, 2006, 2005, 2006]

df["data1"].groupby([states, years]).mean()

CA  2005    0.024273
    2006    0.576845
OH  2005    1.078754
    2006    0.833280
Name: data1, dtype: float64

Frequently, the grouping information is found in the same DataFrame as the data you want to work on. In that case, you can pass column names (whether those are strings, numbers, or other Python objects) as the group keys:

In [ ]:
df.groupby("key1").mean()

,key2,data1,data2
key1,,,
a,1.5,0.56203,-0.159429
b,1.5,0.41331,-0.640091


In [ ]:
df.groupby("key2").mean(numeric_only=True)

,data1,data2
key2,,
1,0.970237,-0.792111
2,-0.216592,-0.390844


In [ ]:
df.groupby(["key1", "key2"]).mean()

data1     data2
key1 key2                    
a    1     1.637544 -0.305685
     2    -0.953149 -0.332670
b    1     0.306656 -0.831162
     2     0.519965 -0.449019

> You may notice that in the second case, it is necessary to pass numeric_only=True because the key1 column is not numeric and thus cannot be aggregated with mean().

Regardless of the objective in using groupby, a generally useful GroupBy method is `size`, which returns a Series containing group sizes:

In [ ]:
# 相当于 SQL 的 count()
df.groupby(["key1", "key2"]).size()

key1  key2
a     1       1
      2       1
b     1       1
      2       1
dtype: int64

Note that any missing values in a group key are excluded from the result by default. This behavior can be disabled by passing `dropna=False` to `groupby`:

In [ ]:
df.groupby("key1", dropna=False).size()

,0
key1,
a,3
b,2
NaN,2


In [ ]:
df.groupby(["key1", "key2"], dropna=False).size()

key1  key2
a     1       1
      2       1
      <NA>    1
b     1       1
      2       1
NaN   1       2
dtype: int64

A group function similar in spirit to size is `count`, which computes the number of nonnull values in each group:

In [ ]:
df.groupby("key1").count()

,key2,data1,data2
key1,,,
a,2,3,3
b,2,2,2


## 1.1 Iterating over Groups

The object returned by groupby supports iteration, generating a sequence of 2-tuples containing the group name along with the chunk of data. Consider the following:

In [ ]:
for name, group in df.groupby("key1"):
  print(name)
  print(group)

a
  key1  key2     data1     data2
0    a     1  1.637544 -0.305685
1    a     2 -0.953149 -0.332670
5    a  <NA>  1.001695  0.160066
b
  key1  key2     data1     data2
3    b     2  0.519965 -0.449019
4    b     1  0.306656 -0.831162


In the case of multiple keys, the first element in the tuple will be a tuple of key values:



In [ ]:
for (k1, k2), group in df.groupby(["key1", "key2"]):
  print((k1, k2))
  print(group)

('a', np.int64(1))
  key1  key2     data1     data2
0    a     1  1.637544 -0.305685
('a', np.int64(2))
  key1  key2     data1    data2
1    a     2 -0.953149 -0.33267
('b', np.int64(1))
  key1  key2     data1     data2
4    b     1  0.306656 -0.831162
('b', np.int64(2))
  key1  key2     data1     data2
3    b     2  0.519965 -0.449019


Of course, you can choose to do whatever you want with the pieces of data. A recipe you may find useful is computing a dictionary of the data pieces as a one-liner:

In [ ]:
pieces = {name: group for name, group in df.groupby("key1")}

pieces["b"]

,key1,key2,data1,data2
3,b,2,0.519965,-0.449019
4,b,1,0.306656,-0.831162


By default groupby groups on axis="index", but you can group on any of the other axes. For example, we could group the columns of our example df here by whether they start with `"key"` or `"data"`:

In [ ]:
grouped = df.groupby({"key1": "key", "key2": "key", "data1": "data", "data2": "data"}, axis="columns")

for group_key, group_value in grouped:
  print(group_key)
  print(group_value)

data
      data1     data2
0  1.637544 -0.305685
1 -0.953149 -0.332670
2  0.576845 -1.431820
3  0.519965 -0.449019
4  0.306656 -0.831162
5  1.001695  0.160066
6  1.359904 -0.599777
key
   key1  key2
0     a     1
1     a     2
2  None     1
3     b     2
4     b     1
5     a  <NA>
6  None     1


/tmp/ipython-input-353563112.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  grouped = df.groupby({"key1": "key", "key2": "key", "data1": "data", "data2": "data"}, axis="columns")


## 1.2 Selecting a Column or Subset of Columns

Indexing a GroupBy object created from a DataFrame with a column name or array of column names has the effect of column subsetting for aggregation. This means that:



In [ ]:
df.groupby("key1")["data1"].mean()

,data1
key1,
a,0.56203
b,0.41331


Especially for large datasets, it may be desirable to aggregate only a few columns. For example, in the preceding dataset, to compute the means for just the data2 column and get the result as a DataFrame, we could write:

In [ ]:
# [["data2"]] 如果传入的是 list or array，返回的就是 dataframe
result = (df.groupby(["key1", "key2"])[["data2"]]
          .mean()
          .round(2)
          .rename(columns={"data2": "平均值"}))

print(result)

            平均值
key1 key2      
a    1    -0.31
     2    -0.33
b    1    -0.83
     2    -0.45


Return a grouped Series if only a single column name is passed as a scalar

In [ ]:
s_grouped = df.groupby(["key1", "key2"])["data2"]

s_grouped

In [ ]:
s_grouped.mean()

key1  key2
a     1      -0.305685
      2      -0.332670
b     1      -0.831162
      2      -0.449019
Name: data2, dtype: float64

## 1.3 Grouping with Dictionaries and Series

In [ ]:
people = pd.DataFrame(np.random.standard_normal((5, 5)),
                       columns=["a", "b", "c", "d", "e"],
                       index=["Joe", "Steve", "Wanda", "Jill", "Trey"])

people.iloc[2:3, [1, 2]] = np.nan # Add a few NA values

people

,a,b,c,d,e
Joe,0.654624,0.757916,0.348641,-0.010151,0.220128
Steve,0.533941,-0.347018,0.159320,0.614434,1.011216
Wanda,0.229347,NaN,NaN,-0.690094,0.162865
Jill,0.926843,-2.603335,-0.464867,-0.525662,0.074703
Trey,-0.460393,-0.443107,0.269683,0.426215,-1.151093


Now, suppose I have a group correspondence for the columns and want to sum the columns by group:

In [ ]:
mapping = {"a": "red", "b": "red", "c": "blue", "d": "blue", "e": "red", "f" : "orange"}

people.groupby(mapping, axis="columns").sum()

/tmp/ipython-input-1388268510.py:3: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  people.groupby(mapping, axis="columns").sum()


,blue,red
Joe,0.338489,1.632668
Steve,0.773755,1.198139
Wanda,-0.690094,0.392212
Jill,-0.990529,-1.601789
Trey,0.695898,-2.054592


## 1.4 Grouping with Functions

Suppose you wanted to group by name length. While you could compute an array of string lengths, it's simpler to just pass the `len` function:

In [ ]:
people.groupby(len).sum()

,a,b,c,d,e
3,0.654624,0.757916,0.348641,-0.010151,0.220128
4,0.466450,-3.046442,-0.195184,-0.099448,-1.076390
5,0.763289,-0.347018,0.159320,-0.075660,1.174081


Mixing functions with arrays, dictionaries, or Series is not a problem, as everything gets converted to arrays internally:

In [ ]:
key_list = ["one", "one", "one", "two", "two"]

people.groupby([len, key_list]).min()

,,a,b,c,d,e
3,one,0.654624,0.757916,0.348641,-0.010151,0.220128
4,two,-0.460393,-2.603335,-0.464867,-0.525662,-1.151093
5,one,0.229347,-0.347018,0.159320,-0.690094,0.162865


## 1.5 Grouping by Index Levels

A final convenience for hierarchically indexed datasets is the ability to aggregate using one of the levels of an axis index. Let's look at an example:

In [ ]:
columns = pd.MultiIndex.from_arrays([["US", "US", "US", "JP", "JP"],
                                     [1, 3, 5, 1, 3]],
                                     names=["cty", "tenor"])

hier_df = pd.DataFrame(np.random.standard_normal((4, 5)), columns=columns)

hier_df

cty          US                            JP          
tenor         1         3         5         1         3
0     -1.084612 -1.078627  0.148841 -1.029319  1.513759
1      0.039090  0.825206 -0.179427 -0.816540  0.844137
2     -0.212598  1.742214 -1.234068  1.976283  1.801509
3     -1.476359  0.314066  0.990801 -0.769732  0.509676

To group by level, pass the level number or name using the `level` keyword:

In [ ]:
hier_df.groupby(level="cty", axis="columns").count()

/tmp/ipython-input-2795167867.py:1: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  hier_df.groupby(level="cty", axis="columns").count()


cty,JP,US
0,2,3
1,2,3
2,2,3
3,2,3


# 2 Data Aggregation

In [ ]:
df

,key1,key2,data1,data2
0,a,1,1.637544,-0.305685
1,a,2,-0.953149,-0.332670
2,None,1,0.576845,-1.431820
3,b,2,0.519965,-0.449019
4,b,1,0.306656,-0.831162
5,a,<NA>,1.001695,0.160066
6,None,1,1.359904,-0.599777


In [ ]:
# nsmallest() 方法用于获取每个分组中最小的 n 个值
df.groupby("key1")["data1"].nsmallest(2)

key1   
a     1   -0.953149
      5    1.001695
b     4    0.306656
      3    0.519965
Name: data1, dtype: float64

To use your own aggregation functions, pass any function that aggregates an array to the aggregate method or its short alias `agg`:

In [ ]:
def peak_to_peak(arr):
  return arr.max() - arr.min()

df.groupby("key1").agg(peak_to_peak)

,key2,data1,data2
key1,,,
a,1,2.590692,0.492736
b,1,0.213309,0.382143


You may notice that some methods, like describe, also work, even though they are not aggregations, strictly speaking

In [ ]:
df.groupby("key1").describe()

key2                                           data1           ...  \
     count mean       std  min   25%  50%   75%  max count     mean  ...   
key1                                                                 ...   
a      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   3.0  0.56203  ...   
b      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   2.0  0.41331  ...   

                         data2                                          \
           75%       max count      mean       std       min       25%   
key1                                                                     
a     1.319619  1.637544   3.0 -0.159429  0.277020 -0.332670 -0.319177   
b     0.466638  0.519965   2.0 -0.640091  0.270216 -0.831162 -0.735626   

                                    
           50%       75%       max  
key1                                
a    -0.305685 -0.072809  0.160066  
b    -0.640091 -0.544555 -0.449019  

[2 rows x 24 columns]

## 2.1 Column-Wise and Multiple Function Application

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
tips = pd.read_csv("/content/drive/My Drive/Colab Notebooks/Python for Data Analysis/examples/tips.csv")

tips.head()

,total_bill,tip,smoker,day,time,size
0,16.99,1.01,No,Sun,Dinner,2
1,10.34,1.66,No,Sun,Dinner,3
2,21.01,3.50,No,Sun,Dinner,3
3,23.68,3.31,No,Sun,Dinner,2
4,24.59,3.61,No,Sun,Dinner,4


Now I will add a `tip_pct` column with the tip percentage of the total bill:

In [ ]:
tips["tip_pct"] = tips["tip"] / tips["total_bill"]

tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


you may want to aggregate using a different function, depending on the column, or multiple functions at once. Fortunately, this is possible to do, which I’ll illustrate through a number of examples. First, I’ll group the tips by day and smoker:

```python
# .mean() - 单一功能
result_simple = tips.groupby(["day", "smoker"])["tip_pct"].mean()

# .agg() - 支持多个聚合函数
result_multi = tips.groupby(["day", "smoker"])["tip_pct"].agg(["mean", "std", "min", "max"])
```

In [ ]:
tips.groupby(["day", "smoker"])["tip_pct"].agg(["mean", "std", "min", "max", peak_to_peak])

mean       std       min       max  peak_to_peak
day  smoker                                                      
Fri  No      0.151650  0.028123  0.120385  0.187735      0.067349
     Yes     0.174783  0.051293  0.103555  0.263480      0.159925
Sat  No      0.158048  0.039767  0.056797  0.291990      0.235193
     Yes     0.147906  0.061375  0.035638  0.325733      0.290095
Sun  No      0.160113  0.042347  0.059447  0.252672      0.193226
     Yes     0.187250  0.154134  0.065660  0.710345      0.644685
Thur No      0.160298  0.038774  0.072961  0.266312      0.193350
     Yes     0.163863  0.039389  0.090014  0.241255      0.151240

 if you pass a list of `(name, function)` tuples, the first element of each tuple will be used as the DataFrame column names

In [ ]:
tips.groupby(["day", "smoker"])["tip_pct"].agg([("average", "mean"), ("stdev", np.std)])

/tmp/ipython-input-748620616.py:1: FutureWarning: The provided callable <function std at 0x7f876b178360> is currently using SeriesGroupBy.std. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "std" instead.
  tips.groupby(["day", "smoker"])["tip_pct"].agg([("average", "mean"), ("stdev", np.std)])


average     stdev
day  smoker                    
Fri  No      0.151650  0.028123
     Yes     0.174783  0.051293
Sat  No      0.158048  0.039767
     Yes     0.147906  0.061375
Sun  No      0.160113  0.042347
     Yes     0.187250  0.154134
Thur No      0.160298  0.038774
     Yes     0.163863  0.039389

With a DataFrame you have more options, as you can specify a list of functions to apply to all of the columns or different functions per column. To start, suppose we wanted to compute the same three statistics for the `tip_pct` and `total_bill` columns:

In [ ]:
functions = ["count", "mean", "max"]

result = tips.groupby(["day", "smoker"])[["tip_pct","total_bill"]].agg(functions)

result

tip_pct                     total_bill                  
              count      mean       max      count       mean    max
day  smoker                                                         
Fri  No           4  0.151650  0.187735          4  18.420000  22.75
     Yes         15  0.174783  0.263480         15  16.813333  40.17
Sat  No          45  0.158048  0.291990         45  19.661778  48.33
     Yes         42  0.147906  0.325733         42  21.276667  50.81
Sun  No          57  0.160113  0.252672         57  20.506667  48.17
     Yes         19  0.187250  0.710345         19  24.120000  45.35
Thur No          45  0.160298  0.266312         45  17.113111  41.19
     Yes         17  0.163863  0.241255         17  19.190588  43.11

As before, a list of tuples with custom names can be passed:

In [ ]:
ftuples = [("Average", "mean"), ("Variance", np.var)]

tips.groupby(["day", "smoker"])[["tip_pct","total_bill"]].agg(ftuples)

/tmp/ipython-input-384228376.py:3: FutureWarning: The provided callable <function var at 0x7f876b1784a0> is currently using SeriesGroupBy.var. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "var" instead.
  tips.groupby(["day", "smoker"])[["tip_pct","total_bill"]].agg(ftuples)


tip_pct           total_bill            
              Average  Variance    Average    Variance
day  smoker                                           
Fri  No      0.151650  0.000791  18.420000   25.596333
     Yes     0.174783  0.002631  16.813333   82.562438
Sat  No      0.158048  0.001581  19.661778   79.908965
     Yes     0.147906  0.003767  21.276667  101.387535
Sun  No      0.160113  0.001793  20.506667   66.099980
     Yes     0.187250  0.023757  24.120000  109.046044
Thur No      0.160298  0.001503  17.113111   59.625081
     Yes     0.163863  0.001551  19.190588   69.808518

Now, suppose you wanted to apply potentially different functions to one or more of the columns. To do this, pass a dictionary to `agg` that contains a mapping of column names to any of the function specifications listed so far:

In [ ]:
tips.groupby(["day", "smoker"]).agg({"tip": np.max, "size": "sum"})

/tmp/ipython-input-1097996335.py:1: FutureWarning: The provided callable <function max at 0x7f876b16f7e0> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  tips.groupby(["day", "smoker"]).agg({"tip": np.max, "size": "sum"})


tip  size
day  smoker             
Fri  No       3.50     9
     Yes      4.73    31
Sat  No       9.00   115
     Yes     10.00   104
Sun  No       6.00   167
     Yes      6.50    49
Thur No       6.70   112
     Yes      5.00    40

In [ ]:
tips.groupby(["day", "smoker"]).agg({"tip_pct": ["min", "max", "mean", "std"],
                                     "size": "sum"})

tip_pct                               size
                  min       max      mean       std  sum
day  smoker                                             
Fri  No      0.120385  0.187735  0.151650  0.028123    9
     Yes     0.103555  0.263480  0.174783  0.051293   31
Sat  No      0.056797  0.291990  0.158048  0.039767  115
     Yes     0.035638  0.325733  0.147906  0.061375  104
Sun  No      0.059447  0.252672  0.160113  0.042347  167
     Yes     0.065660  0.710345  0.187250  0.154134   49
Thur No      0.072961  0.266312  0.160298  0.038774  112
     Yes     0.090014  0.241255  0.163863  0.039389   40

## 2.2 Returning Aggregated Data Without Row Indexes

In all of the examples up until now, the aggregated data comes back with an index, potentially hierarchical, composed from the unique group key combinations. Since this isn’t always desirable, you can disable this behavior in most cases by passing `as_index=False` to `groupby`:

In [ ]:
grouped = tips.groupby(["day", "smoker"], as_index=False)

grouped.mean(numeric_only=True)

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863


# 3 Apply: General split-apply-combine

Returning to the tipping dataset from before, suppose you wanted to select the top five `tip_pct` values by group. First, write a function that selects the rows with the largest values in a particular column:

In [ ]:
def top(df, n=5, column="tip_pct"):
  return df.sort_values(column, ascending=False)[:n]

top(tips, n=6)

,total_bill,tip,smoker,day,time,size,tip_pct
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
232,11.61,3.39,No,Sat,Dinner,2,0.291990
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525


Now, if we group by smoker, say, and call apply with this function, we get the following:

In [ ]:
tips.groupby("smoker").apply(top)

/tmp/ipython-input-2530541573.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby("smoker").apply(top)


total_bill   tip smoker   day    time  size   tip_pct
smoker                                                           
No     232       11.61  3.39     No   Sat  Dinner     2  0.291990
       149        7.51  2.00     No  Thur   Lunch     2  0.266312
       51        10.29  2.60     No   Sun  Dinner     2  0.252672
       185       20.69  5.00     No   Sun  Dinner     5  0.241663
       88        24.71  5.85     No  Thur   Lunch     2  0.236746
Yes    172        7.25  5.15    Yes   Sun  Dinner     2  0.710345
       178        9.60  4.00    Yes   Sun  Dinner     2  0.416667
       67         3.07  1.00    Yes   Sat  Dinner     1  0.325733
       183       23.17  6.50    Yes   Sun  Dinner     4  0.280535
       109       14.31  4.00    Yes   Sat  Dinner     2  0.279525

If you pass a function to apply that takes other arguments or keywords, you can pass these after the function:

In [ ]:
tips.groupby(["smoker", "day"]).apply(top, n=2, column="total_bill")

/tmp/ipython-input-2728870256.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby(["smoker", "day"]).apply(top, n=2, column="total_bill")


total_bill    tip smoker   day    time  size   tip_pct
smoker day                                                             
No     Fri  94        22.75   3.25     No   Fri  Dinner     2  0.142857
            91        22.49   3.50     No   Fri  Dinner     2  0.155625
       Sat  212       48.33   9.00     No   Sat  Dinner     4  0.186220
            59        48.27   6.73     No   Sat  Dinner     4  0.139424
       Sun  156       48.17   5.00     No   Sun  Dinner     6  0.103799
            112       38.07   4.00     No   Sun  Dinner     3  0.105070
       Thur 142       41.19   5.00     No  Thur   Lunch     5  0.121389
            85        34.83   5.17     No  Thur   Lunch     4  0.148435
Yes    Fri  95        40.17   4.73    Yes   Fri  Dinner     4  0.117750
            90        28.97   3.00    Yes   Fri  Dinner     2  0.103555
       Sat  170       50.81  10.00    Yes   Sat  Dinner     3  0.196812
            102       44.30   2.50    Yes   Sat  Dinner     3  0.056433
       Sun  182       45.35   3.50    Yes   Sun  Dinner     3  0.077178
            184       40.55   3.00    Yes   Sun  Dinner     2  0.073983
       Thur 197       43.11   5.00    Yes  Thur   Lunch     4  0.115982
            83        32.68   5.00    Yes  Thur   Lunch     2  0.152999

Beyond these basic usage mechanics, getting the most out of apply may require some creativity. What occurs inside the function passed is up to you; it must either return a pandas object or a scalar value. The rest of this chapter will consist mainly of examples showing you how to solve various problems using groupby.

For example, you may recall that I earlier called `describe` on a GroupBy object:

In [ ]:
result = tips.groupby("smoker")["tip_pct"].describe()

result

,count,mean,std,min,25%,50%,75%,max
smoker,,,,,,,,
No,151.0,0.159328,0.039910,0.056797,0.136906,0.155625,0.185014,0.291990
Yes,93.0,0.163196,0.085119,0.035638,0.106771,0.153846,0.195059,0.710345


In [ ]:
result.unstack("smoker")

smoker
count  No        151.000000
       Yes        93.000000
mean   No          0.159328
       Yes         0.163196
std    No          0.039910
       Yes         0.085119
min    No          0.056797
       Yes         0.035638
25%    No          0.136906
       Yes         0.106771
50%    No          0.155625
       Yes         0.153846
75%    No          0.185014
       Yes         0.195059
max    No          0.291990
       Yes         0.710345
dtype: float64

`group_keys=False` 用于抑制分组键的显示，避免在结果中创建多层索引。

In [ ]:
tips.groupby("smoker").apply(top)

/tmp/ipython-input-2530541573.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby("smoker").apply(top)


total_bill   tip smoker   day    time  size   tip_pct
smoker                                                           
No     232       11.61  3.39     No   Sat  Dinner     2  0.291990
       149        7.51  2.00     No  Thur   Lunch     2  0.266312
       51        10.29  2.60     No   Sun  Dinner     2  0.252672
       185       20.69  5.00     No   Sun  Dinner     5  0.241663
       88        24.71  5.85     No  Thur   Lunch     2  0.236746
Yes    172        7.25  5.15    Yes   Sun  Dinner     2  0.710345
       178        9.60  4.00    Yes   Sun  Dinner     2  0.416667
       67         3.07  1.00    Yes   Sat  Dinner     1  0.325733
       183       23.17  6.50    Yes   Sun  Dinner     4  0.280535
       109       14.31  4.00    Yes   Sat  Dinner     2  0.279525

In [ ]:
tips.groupby("smoker", group_keys=False).apply(top)

/tmp/ipython-input-3851957478.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tips.groupby("smoker", group_keys=False).apply(top)


,total_bill,tip,smoker,day,time,size,tip_pct
232,11.61,3.39,No,Sat,Dinner,2,0.291990
149,7.51,2.00,No,Thur,Lunch,2,0.266312
51,10.29,2.60,No,Sun,Dinner,2,0.252672
185,20.69,5.00,No,Sun,Dinner,5,0.241663
88,24.71,5.85,No,Thur,Lunch,2,0.236746
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525


## 3.1 Quantile and Bucket Analysis

As you may recall from Ch 8: Data Wrangling: Join, Combine, and Reshape, pandas has some tools, in particular pandas.cut and pandas.qcut, for slicing data up into buckets with bins of your choosing, or by sample quantiles. Combining these functions with groupby makes it convenient to perform bucket or quantile analysis on a dataset. Consider a simple random dataset and an equal-length bucket categorization using `pandas.cut`:

In [ ]:
frame = pd.DataFrame({"data1": np.random.standard_normal(1000),
                       "data2": np.random.standard_normal(1000)})

frame.head()

,data1,data2
0,0.795067,1.023530
1,1.183534,1.025140
2,-2.109163,-0.339816
3,-1.042102,1.512594
4,0.744673,0.724002


In [ ]:
quartiles = pd.cut(frame["data1"], 4)

In [ ]:
quartiles.head(10)

,data1
0,"(0.0725, 1.577]"
1,"(0.0725, 1.577]"
2,"(-2.943, -1.432]"
3,"(-1.432, 0.0725]"
4,"(0.0725, 1.577]"
5,"(-1.432, 0.0725]"
6,"(-2.943, -1.432]"
7,"(-1.432, 0.0725]"
8,"(-1.432, 0.0725]"
9,"(-1.432, 0.0725]"


The `Categorical` object returned by `cut` can be passed directly to `groupby`. So we could compute a set of group statistics for the quartiles, like so:

In [ ]:
def get_stats(group):
  return pd.DataFrame({
      "min": group.min(),
      "max": group.max(),
      "count": group.count(),
      "mean": group.mean()
  })

grouped = frame.groupby(quartiles)

grouped.apply(get_stats)

/tmp/ipython-input-315112711.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = frame.groupby(quartiles)


min       max  count      mean
data1                                                      
(-2.943, -1.432] data1 -2.937431 -1.436894     73 -1.871979
                 data2 -2.236428  2.109471     73 -0.240243
(-1.432, 0.0725] data1 -1.430664  0.072069    446 -0.590590
                 data2 -3.297935  2.546590    446 -0.045779
(0.0725, 1.577]  data1  0.074110  1.568413    427  0.668624
                 data2 -3.341820  2.894564    427  0.011792
(1.577, 3.082]   data1  1.578956  3.082425     54  1.984163
                 data2 -3.649418  2.037675     54 -0.017256

Keep in mind the same result could have been computed more simply with:

In [ ]:
grouped.agg(["min", "max", "count", "mean"])

data1                               data2            \
                       min       max count      mean       min       max   
data1                                                                      
(-2.943, -1.432] -2.937431 -1.436894    73 -1.871979 -2.236428  2.109471   
(-1.432, 0.0725] -1.430664  0.072069   446 -0.590590 -3.297935  2.546590   
(0.0725, 1.577]   0.074110  1.568413   427  0.668624 -3.341820  2.894564   
(1.577, 3.082]    1.578956  3.082425    54  1.984163 -3.649418  2.037675   

                                  
                 count      mean  
data1                             
(-2.943, -1.432]    73 -0.240243  
(-1.432, 0.0725]   446 -0.045779  
(0.0725, 1.577]    427  0.011792  
(1.577, 3.082]      54 -0.017256

These were equal-length buckets; to compute equal-size buckets based on sample quantiles, use pandas.qcut. We can pass 4 as the number of bucket compute sample quartiles, and pass `labels=False` to obtain just the quartile indices instead of intervals:

In [ ]:
quartiles_samp = pd.qcut(frame["data1"], 4)

quartiles_samp.head()

,data1
0,"(0.677, 3.082]"
1,"(0.677, 3.082]"
2,"(-2.9379999999999997, -0.704]"
3,"(-2.9379999999999997, -0.704]"
4,"(0.677, 3.082]"


In [ ]:
quartiles_samp = pd.qcut(frame["data1"], 4, labels=False)

quartiles_samp.head()

,data1
0,3
1,3
2,0
3,0
4,3


In [ ]:
grouped = frame.groupby(quartiles_samp)

grouped.apply(get_stats)

min       max  count      mean
data1                                           
0     data1 -2.937431 -0.704265    250 -1.269169
      data2 -3.293478  2.288379    250 -0.032305
1     data1 -0.703350  0.017805    250 -0.334572
      data2 -3.297935  2.546590    250 -0.124696
2     data1  0.022831  0.675926    250  0.339590
      data2 -2.956163  2.358535    250  0.029735
3     data1  0.681942  3.082425    250  1.234510
      data2 -3.649418  2.894564    250 -0.008142

## 3.2 Example: Filling Missing Values with Group-Specific Values

When cleaning up missing data, in some cases you will remove data observations using dropna, but in others you may want to fill in the null (NA) values using a fixed value or some value derived from the data. `fillna` is the right tool to use; for example, here I fill in the null values with the mean:

In [ ]:
s = pd.Series(np.random.standard_normal(6))

s[::2] = np.nan

s

,0
0,NaN
1,0.297005
2,NaN
3,-0.245132
4,NaN
5,-1.179152


In [ ]:
s.fillna(s.mean())

,0
0,-0.375760
1,0.297005
2,-0.375760
3,-0.245132
4,-0.375760
5,-1.179152


Suppose you need the fill value to vary by group. One way to do this is to group the data and use apply with a function that calls `fillna` on each data chunk. Here is some sample data on US states divided into eastern and western regions:

In [ ]:
states = ["Ohio", "New York", "Vermont", "Florida", "Oregon", "Nevada", "California", "Idaho"]

group_key = ["East", "East", "East", "East", "West", "West", "West", "West"]

data = pd.Series(np.random.standard_normal(8), index=states)

data

,0
Ohio,-0.482018
New York,-0.055356
Vermont,-0.098888
Florida,-1.495304
Oregon,1.005826
Nevada,1.468804
California,-1.070166
Idaho,0.040983


Let's set some values in the data to be missing:

In [ ]:
data[["Vermont", "Nevada", "Idaho"]] = np.nan

data

,0
Ohio,-0.482018
New York,-0.055356
Vermont,NaN
Florida,-1.495304
Oregon,1.005826
Nevada,NaN
California,-1.070166
Idaho,NaN


In [ ]:
data.groupby(group_key).size()

,0
East,4
West,4


In [ ]:
data.groupby(group_key).count()

,0
East,3
West,2


In [ ]:
data.groupby(group_key).mean()

,0
East,-0.677559
West,-0.032170


We can fill the NA values using the group means, like so:

In [ ]:
def fill_mean(group):
  return group.fillna(group.mean())

data.groupby(group_key).apply(fill_mean)

East  Ohio         -0.482018
      New York     -0.055356
      Vermont      -0.677559
      Florida      -1.495304
West  Oregon        1.005826
      Nevada       -0.032170
      California   -1.070166
      Idaho        -0.032170
dtype: float64

In another case, you might have predefined fill values in your code that vary by group. Since the groups have a `name` attribute set internally, we can use that:

In [ ]:
fill_values = {"East": 0.5, "West": -1}

def fill_func(group):
  return group.fillna(fill_values[group.name])

data.groupby(group_key).apply(fill_func)

East  Ohio         -0.482018
      New York     -0.055356
      Vermont       0.500000
      Florida      -1.495304
West  Oregon        1.005826
      Nevada       -1.000000
      California   -1.070166
      Idaho        -1.000000
dtype: float64

## 3.3 Example: Random Sampling and Permutation

Suppose you wanted to draw a random sample (with or without replacement) from a large dataset for Monte Carlo simulation purposes or some other application. There are a number of ways to perform the “draws”; here we use the `sample` method for Series.

To demonstrate, here’s a way to construct a deck of English-style playing cards:

In [ ]:
suits = ["H", "S", "C", "D"]  # Hearts, Spades, Clubs, Diamonds

card_val = (list(range(1, 11)) + [10] * 3) * 4

base_names = ["A"] + list(range(2, 11)) + ["J", "K", "Q"]

cards = []

for suit in suits:
    cards.extend(str(num) + suit for num in base_names)

deck = pd.Series(card_val, index=cards)

Now we have a Series of length 52 whose index contains card names, and values are the ones used in blackjack and other games (to keep things simple, I let the ace "A" be 1):

In [ ]:
deck.head(13)

,0
AH,1
2H,2
3H,3
4H,4
5H,5
6H,6
7H,7
8H,8
9H,9
10H,10


Now, based on what I said before, drawing a hand of five cards from the deck could be written as:

In [ ]:
def draw(deck, n=5):
  return deck.sample(n)

draw(deck)

,0
8H,8
8D,8
2H,2
3H,3
7D,7


Suppose you wanted two random cards from each suit. Because the suit is the last character of each card name, we can group based on this and use apply:

In [ ]:
def get_suit(card):
  return card[-1]

deck.groupby(get_suit).apply(draw, n=2)

C  3C      3
   10C    10
D  5D      5
   QD     10
H  5H      5
   9H      9
S  2S      2
   9S      9
dtype: int64

Alternatively, we could pass group_keys=False to drop the outer suit index, leaving in just the selected cards:

In [ ]:
deck.groupby(get_suit, group_keys=False).apply(draw, n=2)

,0
4C,4
8C,8
5D,5
QD,10
AH,1
6H,6
KS,10
JS,10


## 3.4 Example: Group Weighted Average and Correlation

Under the split-apply-combine paradigm of groupby, operations between columns in a DataFrame or two Series, such as a group weighted average, are possible. As an example, take this dataset containing group keys, values, and some weights:

In [ ]:
df = pd.DataFrame({"category": ["a", "a", "a", "a", "b", "b", "b", "b"],
                    "data": np.random.standard_normal(8),
                    "weights": np.random.uniform(size=8)})

df

,category,data,weights
0,a,1.144650,0.439133
1,a,-0.240438,0.561707
2,a,0.561831,0.798602
3,a,-0.794695,0.907510
4,b,-2.026772,0.681726
5,b,-0.603750,0.064103
6,b,0.618161,0.899910
7,b,-0.858314,0.727328


The weighted average by `category` would then be:

In [ ]:
grouped = df.groupby("category")

def get_wavg(group):
  return np.average(group["data"], weights=group["weights"])

grouped.apply(get_wavg)

/tmp/ipython-input-2828575551.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped.apply(get_wavg)


,0
category,
a,0.035126
b,-0.627202


# 4 Group Transforms and "Unwrapped" GroupBys

we looked at the apply method in grouped operations for performing transformations. There is another built-in method called transform, which is similar to apply but imposes more constraints on the kind of function you can use:

* It can produce a scalar value to be broadcast to the shape of the group.

* It can produce an object of the same shape as the input group.

* It must not mutate its input.

Let's consider a simple example for illustration

In [ ]:
df = pd.DataFrame({
    'key': ['a', 'b', 'c'] * 4,
    'value': np.arange(12.)
})

df

,key,value
0,a,0.0
1,b,1.0
2,c,2.0
3,a,3.0
4,b,4.0
5,c,5.0
6,a,6.0
7,b,7.0
8,c,8.0
9,a,9.0


Here are the group means by key:

In [ ]:

g = df.groupby('key')['value']

g.mean()

,value
key,
a,4.5
b,5.5
c,6.5


Suppose instead we wanted to produce a Series of the same shape as df['value'] but with values replaced by the average grouped by 'key'. We can pass a function that computes the mean of a single group to `transform`:

`transform` 的返回形状必须与原始组相同，而 `apply` 可以是任何形状

In [ ]:
def get_mean(group):
  return group.mean()

g.transform(get_mean)

,value
0,4.5
1,5.5
2,6.5
3,4.5
4,5.5
5,6.5
6,4.5
7,5.5
8,6.5
9,4.5


In [ ]:
g.transform('mean')

,value
0,4.5
1,5.5
2,6.5
3,4.5
4,5.5
5,6.5
6,4.5
7,5.5
8,6.5
9,4.5


Like apply, `transform` works with functions that return Series, but the result must be the same size as the input. For example, we can multiply each group by 2 using a helper function:

In [ ]:
def time_two(group):
  return group * 2

g.transform(time_two)

,value
0,0.0
1,2.0
2,4.0
3,6.0
4,8.0
5,10.0
6,12.0
7,14.0
8,16.0
9,18.0


As a more complicated example, we can compute the ranks in descending order for each group:

In [ ]:
def get_ranks(group):
  return group.rank(ascending=False)

g.transform(get_ranks)

,value
0,4.0
1,4.0
2,4.0
3,3.0
4,3.0
5,3.0
6,2.0
7,2.0
8,2.0
9,1.0


Consider a group transformation function composed from simple aggregations:

In [ ]:
def normalize(x):
  return (x - x.mean()) / x.std()

g.transform(normalize)

,value
0,-1.161895
1,-1.161895
2,-1.161895
3,-0.387298
4,-0.387298
5,-0.387298
6,0.387298
7,0.387298
8,0.387298
9,1.161895


In [ ]:
g.apply(normalize)

key    
a    0    -1.161895
     3    -0.387298
     6     0.387298
     9     1.161895
b    1    -1.161895
     4    -0.387298
     7     0.387298
     10    1.161895
c    2    -1.161895
     5    -0.387298
     8     0.387298
     11    1.161895
Name: value, dtype: float64

Built-in aggregate functions like 'mean' or 'sum' are often much faster than a general apply function. These also have a "fast path" when used with transform. This allows us to perform what is called an unwrapped group operation:

In [ ]:
normalized = (df['value'] - g.transform('mean')) / g.transform('std')

normalized

,value
0,-1.161895
1,-1.161895
2,-1.161895
3,-0.387298
4,-0.387298
5,-0.387298
6,0.387298
7,0.387298
8,0.387298
9,1.161895


# 5 Pivot Tables and Cross-Tabulation

In [ ]:
tips.head()

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808


In [ ]:
tips.pivot_table(index=["day", "smoker"], values=["size", "tip", "tip_pct", "total_bill"])

size       tip   tip_pct  total_bill
day  smoker                                          
Fri  No      2.250000  2.812500  0.151650   18.420000
     Yes     2.066667  2.714000  0.174783   16.813333
Sat  No      2.555556  3.102889  0.158048   19.661778
     Yes     2.476190  2.875476  0.147906   21.276667
Sun  No      2.929825  3.167895  0.160113   20.506667
     Yes     2.578947  3.516842  0.187250   24.120000
Thur No      2.488889  2.673778  0.160298   17.113111
     Yes     2.352941  3.030000  0.163863   19.190588

Now, suppose we want to take the average of only tip_pct and size, and additionally group by time. I’ll put smoker in the table columns and `time` and day in the rows:

In [ ]:
tips.pivot_table(index=["time", "day"], columns="smoker", values=["tip_pct", "size"])

size             tip_pct          
smoker             No       Yes        No       Yes
time   day                                         
Dinner Fri   2.000000  2.222222  0.139622  0.165347
       Sat   2.555556  2.476190  0.158048  0.147906
       Sun   2.929825  2.578947  0.160113  0.187250
       Thur  2.000000       NaN  0.159744       NaN
Lunch  Fri   3.000000  1.833333  0.187735  0.188937
       Thur  2.500000  2.352941  0.160311  0.163863

We could augment this table to include partial totals by passing margins=True. This has the effect of adding All row and column labels, with corresponding values being the group statistics for all the data within a single tier:

In [ ]:
tips.pivot_table(index=["time", "day"], columns="smoker", values=["tip_pct", "size"], margins=True)

size                       tip_pct                    
smoker             No       Yes       All        No       Yes       All
time   day                                                             
Dinner Fri   2.000000  2.222222  2.166667  0.139622  0.165347  0.158916
       Sat   2.555556  2.476190  2.517241  0.158048  0.147906  0.153152
       Sun   2.929825  2.578947  2.842105  0.160113  0.187250  0.166897
       Thur  2.000000       NaN  2.000000  0.159744       NaN  0.159744
Lunch  Fri   3.000000  1.833333  2.000000  0.187735  0.188937  0.188765
       Thur  2.500000  2.352941  2.459016  0.160311  0.163863  0.161301
All          2.668874  2.408602  2.569672  0.159328  0.163196  0.160803

If some combinations are empty (or otherwise NA), you may wish to pass a fill_value:

In [ ]:
tips.pivot_table(index=["time", "size", "smoker"], columns="day",values="tip_pct", fill_value=0)

day                      Fri       Sat       Sun      Thur
time   size smoker                                        
Dinner 1    No      0.000000  0.137931  0.000000  0.000000
            Yes     0.000000  0.325733  0.000000  0.000000
       2    No      0.139622  0.162705  0.168859  0.159744
            Yes     0.171297  0.148668  0.207893  0.000000
       3    No      0.000000  0.154661  0.152663  0.000000
            Yes     0.000000  0.144995  0.152660  0.000000
       4    No      0.000000  0.150096  0.148143  0.000000
            Yes     0.117750  0.124515  0.193370  0.000000
       5    No      0.000000  0.000000  0.206928  0.000000
            Yes     0.000000  0.106572  0.065660  0.000000
       6    No      0.000000  0.000000  0.103799  0.000000
Lunch  1    No      0.000000  0.000000  0.000000  0.181728
            Yes     0.223776  0.000000  0.000000  0.000000
       2    No      0.000000  0.000000  0.000000  0.166005
            Yes     0.181969  0.000000  0.000000  0.158843
       3    No      0.187735  0.000000  0.000000  0.084246
            Yes     0.000000  0.000000  0.000000  0.204952
       4    No      0.000000  0.000000  0.000000  0.138919
            Yes     0.000000  0.000000  0.000000  0.155410
       5    No      0.000000  0.000000  0.000000  0.121389
       6    No      0.000000  0.000000  0.000000  0.173706

## 5.1 Cross-Tabulations: Crosstab

A cross-tabulation (or crosstab for short) is a special case of a pivot table that computes group frequencies. Here is an example:

In [ ]:
from io import StringIO

data = """
Sample  Nationality  Handedness
 1   USA  Right-handed
 2   Japan    Left-handed
 3   USA  Right-handed
 4   Japan    Right-handed
 5   Japan    Left-handed
 6   Japan    Right-handed
 7   USA  Right-handed
 8   USA  Left-handed
 9   Japan    Right-handed
 10  USA  Right-handed
"""

data = pd.read_table(StringIO(data), sep="\s+")

data

<>:17: SyntaxWarning: invalid escape sequence '\s'
<>:17: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-923820993.py:17: SyntaxWarning: invalid escape sequence '\s'
  data = pd.read_table(StringIO(data), sep="\s+")


,Sample,Nationality,Handedness
0,1,USA,Right-handed
1,2,Japan,Left-handed
2,3,USA,Right-handed
3,4,Japan,Right-handed
4,5,Japan,Left-handed
5,6,Japan,Right-handed
6,7,USA,Right-handed
7,8,USA,Left-handed
8,9,Japan,Right-handed
9,10,USA,Right-handed


As part of some survey analysis, we might want to summarize this data by nationality and handedness. You could use pivot_table to do this, but the `pandas.crosstab` function can be more convenient:

In [ ]:
pd.crosstab(data["Nationality"], data["Handedness"], margins=True)

Handedness,Left-handed,Right-handed,All
Nationality,,,
Japan,2,3,5
USA,1,4,5
All,3,7,10


The first two arguments to crosstab can each be an array or Series or a list of arrays. As in the tips data:

In [ ]:
pd.crosstab([tips["time"], tips["day"]], tips["smoker"], margins=True)

smoker        No  Yes  All
time   day                
Dinner Fri     3    9   12
       Sat    45   42   87
       Sun    57   19   76
       Thur    1    0    1
Lunch  Fri     1    6    7
       Thur   44   17   61
All          151   93  244